In [2]:
# CELL 1 — Setup
# Run this cell every session. No kernel restart needed after this cell.
import subprocess, sys, os

# Set env vars FIRST — must happen before any bitsandbytes import
os.environ["BNB_CUDA_VERSION"]  = "128"
os.environ["CUDA_VERSION"]      = "128"
os.environ["LD_LIBRARY_PATH"]   = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH","")

def pip(pkg):
    subprocess.run(f"pip install --quiet {pkg}", shell=True, check=False)

# Only install what Kaggle does NOT already have
# Do NOT touch bitsandbytes, torch, torchvision — Kaggle manages these
print("Installing missing packages only...")
pip("transformers==4.37.2 tokenizers==0.15.2 --no-deps")
pip("peft==0.7.1 --no-deps")
pip("accelerate==0.26.1")
pip("sentence-transformers==2.7.0")
pip("datasets huggingface_hub>=0.23.0")
pip("scipy scikit-learn pandas matplotlib pillow")
print("Done.")

Installing missing packages only...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.9/270.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 66.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.0 MB/s eta 0:00:00
Done.


In [5]:
# CELL 2 — Imports
import os, sys

# Must be set BEFORE importing bitsandbytes
os.environ["BNB_CUDA_VERSION"]  = "128"
os.environ["CUDA_VERSION"]      = "128"
os.environ["LD_LIBRARY_PATH"]   = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH","")

import gc, json, warnings, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy import stats as scipy_stats
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore")

import transformers, peft
print(f"torch        {torch.__version__}")
print(f"CUDA         {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"transformers {transformers.__version__}")
print(f"peft         {peft.__version__}")

# Test bitsandbytes separately — it is sensitive to env var order
try:
    import bitsandbytes as bnb
    print(f"bitsandbytes {bnb.__version__}  OK")
    BNB_OK = True
except Exception as e:
    print(f"bitsandbytes FAILED: {e}")
    print("  Will use float16 instead of 4-bit quantisation.")
    BNB_OK = False

print("All imports OK")

torch        2.10.0+cu128
CUDA         True | Tesla P100-PCIE-16GB
transformers 5.0.0
peft         0.18.1
bitsandbytes FAILED: No module named 'bitsandbytes'
  Will use float16 instead of 4-bit quantisation.
All imports OK


In [6]:
# CELL 3 — Configuration
import os

WORK_DIR  = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
CACHE_DIR = os.path.join(WORK_DIR, "llava_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

CFG = {
    # Model
    "MODEL_ID":            "llava-hf/llava-1.5-7b-hf",
    "LORA_RANK":           16,
    # Training
    "PPO_STEPS":           400,
    "TRAIN_ITEMS":         400,
    "LEARNING_RATE":       1e-5,
    "TEMPERATURE":         0.9,
    "MAX_NEW_TOKENS":      50,
    "EMA_DECAY":           0.9,
    "GRAD_CLIP":           0.5,
    # PPO-Clip
    "PPO_EPSILON":         0.2,
    # Evaluation
    "TEST_ITEMS":          500,
    "EVAL_TEMPERATURE":    0.7,
    # Seeds (8 seeds)
    "SEEDS":               [42, 123, 7, 1, 2, 3, 4, 5],
    # Reward
    "W_TASK":              1.0,
    "W_FACT":              0.5,
    "PENALTY":            -0.5,
    "FGM_THRESHOLD":       0.6,
    "TASK_THRESHOLD":      0.7,
    # Stats
    "CI_LEVEL":            0.95,
}

print("CFG ready:")
for k,v in CFG.items():
    print(f"  {k:<25} {v}")


CFG ready:
  MODEL_ID                  llava-hf/llava-1.5-7b-hf
  LORA_RANK                 16
  PPO_STEPS                 400
  TRAIN_ITEMS               400
  LEARNING_RATE             1e-05
  TEMPERATURE               0.9
  MAX_NEW_TOKENS            50
  EMA_DECAY                 0.9
  GRAD_CLIP                 0.5
  PPO_EPSILON               0.2
  TEST_ITEMS                500
  EVAL_TEMPERATURE          0.7
  SEEDS                     [42, 123, 7, 1, 2, 3, 4, 5]
  W_TASK                    1.0
  W_FACT                    0.5
  PENALTY                   -0.5
  FGM_THRESHOLD             0.6
  TASK_THRESHOLD            0.7
  CI_LEVEL                  0.95


In [7]:
# CELL 4 — Environment Setup
# Downloads model weights once, sets up random seed utility

from huggingface_hub import snapshot_download

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def nuke(*extra_names):
    """Free all known model variables from GPU memory."""
    names = list(extra_names) + [
        "model","base","sm_reinforce","sm_ppoclip","sm_hra",
        "vm","vanilla_model","lora_model"
    ]
    for name in names:
        if name in globals():
            try:    globals()[name].cpu()
            except: pass
            try:    del globals()[name]
            except: pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM: {free/1024**3:.2f} GB free / {total/1024**3:.2f} GB total")

# Download weights once (file transfer only — no GPU)
already_cached = any(
    f.endswith(".safetensors")
    for _, _, files in os.walk(CACHE_DIR)
    for f in files
)

if already_cached:
    print(f"✓ Model weights cached at {CACHE_DIR}")
else:
    print(f"Downloading model weights to {CACHE_DIR} ...")
    print("  (One-time download ~14GB — subsequent runs load from disk)")
    snapshot_download(
        repo_id=CFG["MODEL_ID"],
        cache_dir=CACHE_DIR,
        ignore_patterns=["*.msgpack","*.h5","flax_model*","tf_model*"],
    )
    already_cached = True
    print("✓ Download complete")

print(f"\nWorking dir : {WORK_DIR}")
print(f"Cache dir   : {CACHE_DIR}")
print(f"Cached      : {already_cached}")


  (One-time download ~14GB — subsequent runs load from disk)


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

✓ Download complete

Working dir : /kaggle/working
Cache dir   : /kaggle/working/llava_cache
Cached      : True


In [8]:
# CELL 5 — Model Factory

def _bnb():
    if not BNB_OK:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False,
        bnb_4bit_quant_type="nf4",
    )

def _lora():
    return LoraConfig(
        r=CFG["LORA_RANK"],
        lora_alpha=CFG["LORA_RANK"] * 2,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj","v_proj"],
        task_type="CAUSAL_LM",
    )

def make_lora_model():
    gc.collect(); torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM before load: {free/1024**3:.2f} GB / {total/1024**3:.2f} GB")

    bnb_cfg = _bnb()
    kwargs = dict(
        device_map="auto",
        cache_dir=CACHE_DIR,
        local_files_only=already_cached,
    )
    if bnb_cfg is not None:
        kwargs["quantization_config"] = bnb_cfg
        print("  Loading in 4-bit (bitsandbytes)")
    else:
        kwargs["torch_dtype"] = torch.float16
        print("  Loading in float16 (bitsandbytes unavailable)")

    base  = LlavaForConditionalGeneration.from_pretrained(CFG["MODEL_ID"], **kwargs)
    model = get_peft_model(base, _lora())
    model.print_trainable_parameters()
    free2, _ = torch.cuda.mem_get_info()
    print(f"  VRAM after load:  {free2/1024**3:.2f} GB")
    return model

def make_vanilla_model():
    gc.collect(); torch.cuda.empty_cache()
    bnb_cfg = _bnb()
    kwargs = dict(
        device_map="auto",
        cache_dir=CACHE_DIR,
        local_files_only=already_cached,
    )
    if bnb_cfg is not None:
        kwargs["quantization_config"] = bnb_cfg
    else:
        kwargs["torch_dtype"] = torch.float16

    m = LlavaForConditionalGeneration.from_pretrained(CFG["MODEL_ID"], **kwargs)
    free, _ = torch.cuda.mem_get_info()
    print(f"  Vanilla loaded. VRAM: {free/1024**3:.2f} GB")
    return m

print("Model factory ready.")
print(f"  Quantisation: {'4-bit (bitsandbytes)' if BNB_OK else 'float16 fallback'}")


Model factory ready.
  Quantisation: float16 fallback


In [9]:
# CELL 6 — Processor and FGM

processor = AutoProcessor.from_pretrained(
    CFG["MODEL_ID"],
    cache_dir=CACHE_DIR,
    local_files_only=already_cached,
)
processor.tokenizer.pad_token    = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"

# Factual Grounding Module
_fgm_encoder = SentenceTransformer("all-MiniLM-L6-v2")

def fgm_score(response: str, ground_truth: str) -> float:
    """Cosine similarity between response and ground-truth embeddings."""
    emb = _fgm_encoder.encode(
        [response, ground_truth], convert_to_tensor=True, show_progress_bar=False
    )
    return float(torch.nn.functional.cosine_similarity(
        emb[0].unsqueeze(0), emb[1].unsqueeze(0)
    ).item())

print(f"✓ Processor ready  (vocab: {processor.tokenizer.vocab_size})")
print(f"✓ FGM ready        (all-MiniLM-L6-v2)")
print(f"  FGM test score: {fgm_score('the cat sat', 'the cat sat'):.4f} (should be ~1.0)")


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


ModuleNotFoundError: No module named 'transformers.models.aimv2'